In [ ]:
from pathlib import Path

from chunking import chunk_document_file

doc_path = Path("docs/ace12-user_guide.txt")

chunks = chunk_document_file(
    doc_path,
    chunk_size=4600,
    chunk_overlap=1000,
)

print(f"Total chunks: {len(chunks)}")
print(chunks[3200]["id"])
print(chunks[3200]["metadata"])
print(chunks[3200]["text"])

In [ ]:
from dotenv import load_dotenv

from metadata import enrich_chunk

load_dotenv()

test_chunk = chunks[1400]

enriched_chunk = enrich_chunk(
    chunk=test_chunk,
    client=__import__("openai").OpenAI(),
    model="gpt-5-mini",
)

print(enriched_chunk["id"])
print(enriched_chunk["metadata"])
print(enriched_chunk["text"])

In [ ]:
from chunking import chunk_document

chunks = chunk_document()

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

from chunking import chunk_document_file
from metadata import enrich_chunk

load_dotenv()

SOURCE_FILE = Path("docs/ace12-user_guide.txt")
OUTPUT_FILE = Path("data/enriched_chunks.jsonl")

CHUNK_SIZE = 4600
CHUNK_OVERLAP = 1000

MODEL = "gpt-5-mini"

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
chunks = chunk_document_file(
    path=SOURCE_FILE,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

print(f"Generated {len(chunks)} chunks.")

for chunk in chunks[:2]:
    print("\nID:", chunk["id"])
    print("Metadata:", chunk["metadata"])
    print("Text preview:", chunk["text"][:300])

In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import os
from openai import OpenAI

MAX_WORKERS = 100

def load_processed_ids(output_file):
    processed_ids = set()

    if output_file.exists():
        with output_file.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    processed_ids.add(record["id"])

    return processed_ids


processed_ids = load_processed_ids(OUTPUT_FILE)

remaining_chunks = [
    chunk for chunk in chunks
    if chunk["id"] not in processed_ids
]

print(f"Total chunks: {len(chunks)}")
print(f"Already processed: {len(processed_ids)}")
print(f"Remaining: {len(remaining_chunks)}")


In [ ]:

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def process_chunk(chunk):
    return enrich_chunk(
        chunk=chunk,
        client=client,
        model=MODEL,
    )


try:
    with OUTPUT_FILE.open("a", encoding="utf-8") as output:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

            futures = {
                executor.submit(process_chunk, chunk): chunk["id"]
                for chunk in remaining_chunks
            }

            for completed, future in enumerate(
                as_completed(futures), start=1
            ):
                chunk_id = futures[future]

                try:
                    enriched_chunk = future.result()

                    output.write(
                        json.dumps(enriched_chunk, ensure_ascii=False) + "\n"
                    )
                    output.flush()

                    print(
                        f"[{completed}/{len(futures)}] "
                        f"Completed: {chunk_id}"
                    )

                except Exception as error:
                    print(f"FAILED: {chunk_id} — {error}")

finally:
    client.close()


In [ ]:
records = []

with OUTPUT_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))

print(f"Total enriched chunks: {len(records)}")

# Check IDs
ids = [record["id"] for record in records]
print(f"Unique IDs: {len(set(ids))}")

# Check required fields
required_metadata = {
    "source",
    "page",
    "chunk_index",
    "chapter",
    "subtopic",
    "keywords",
    "entities",
    "description",
}

for record in records:
    assert isinstance(record["text"], str)
    assert isinstance(record["metadata"], dict)
    assert required_metadata.issubset(record["metadata"].keys())

print("Validation successful.")

In [11]:

import json
from pathlib import Path

from modal_compute import EmbeddingService
from embeddings import embed_chunks

INPUT_FILE = Path("data/enriched_chunks.jsonl")
OUTPUT_FILE = Path("data/embedded_chunks.jsonl")

# Load enriched chunks
with INPUT_FILE.open("r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

# Connect to the deployed Modal service
embedding_service = EmbeddingService()

# Generate embeddings
embedded_chunks = embed_chunks(
    chunks=chunks,
    embedding_service=embedding_service,
    batch_size=16,
    inference_batch_size=4,
)

# Save results
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for chunk in embedded_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Total embedded chunks: {len(embedded_chunks)}")
print(
    "Embedding dimensions:",
    len(embedded_chunks[0]["embedding"])
)


ExecutionError: Function has not been hydrated with the metadata it needs to run on Modal, because the App it is defined on is not running.

In [ ]:
%pip install -q modal

In [1]:
import modal

modal.is_local()

True

In [6]:
!uv run modal token list

Usage: modal token [OPTIONS] COMMAND [ARGS]...
Try 'modal token -h' for help.
┌─ Error ─────────────────────────────────────────────────────────────────────┐
│ No such command 'list'.                                                     │
└─────────────────────────────────────────────────────────────────────────────┘


In [7]:
!uv run modal setup

The web browser should have opened for you to authenticate and get an API 
token.
If it didn't, please copy this URL into your web browser manually:

⠋ Waiting for authentication in the web browser
https://modal.com/token-flow/tf-r0KJwW2ANpwZTnOAFlZhKd

⠋ Waiting for authentication in the web browser
⠋ Waiting for authentication in the web browser

⠋ Waiting for token flow to complete...
⠋ Waiting for token flow to complete...
⠹ Waiting for token flow to complete...
⠸ Waiting for token flow to complete...
⠼ Waiting for token flow to complete...
⠴ Waiting for token flow to complete...
⠦ Waiting for token flow to complete...
⠧ Waiting for token flow to complete...
⠇ Waiting for token flow to complete...
⠙ Waiting for token flow to complete...
⠹ Waiting for token flow to complete...
⠸ Waiting for token flow to complete...
⠼ Waiting for token flow to complete...
⠦ Waiting for token flow to complete...
⠧ Waiting for token flow to complete...
⠏ Waiting for token flow to complete...
⠋ Waiting

In [8]:
import modal

print("Modal version:", modal.__version__)
print("Modal SDK imported successfully.")

Modal version: 1.2.1
Modal SDK imported successfully.


In [9]:
!uv run modal deploy modal_compute.py

- Creating objects...
| Creating objects...
└── - Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_compute.py: Uploaded
    0/1 files
/ Creating objects...
└── \ Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_compute.py: Uploaded
    0/1 files
| Creating objects...
└── / Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_compute.py: Uploaded
    0/1 files
- Creating objects...
└── | Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_compute.py: Uploaded
    0/1 files
| Creating objects...
└── - Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_compute.py: 
    Finalizing index of 1 files
Building image im-hDPUuqu6azGMWJazev5pHT
| Creating objects...
└── - Creating mount c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_com

=> Step 0: FROM base
/ Creating objects...
└── 🔨 Created mount c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\modal_com

=> Step 1: 

The web browser should have opened for you to authenticate and get an API 
token.
If it didn't, please copy this URL into your web browser manually:

⠋ Waiting for authentication in the web browser
https://modal.com/token-flow/tf-gVpWn66CKpFKSxWDuZ0Xvg

⠋ Waiting for authentication in the web browser
⠋ Waiting for authentication in the web browser

⠋ Waiting for token flow to complete...
⠙ Waiting for token flow to complete...
⠹ Waiting for token flow to complete...
⠸ Waiting for token flow to complete...
⠼ Waiting for token flow to complete...
⠴ Waiting for token flow to complete...
⠦ Waiting for token flow to complete...
⠧ Waiting for token flow to complete...
⠇ Waiting for token flow to complete...
⠏ Waiting for token flow to complete...
⠹ Waiting for token flow to complete...
⠸ Waiting for token flow to complete...
⠼ Waiting for token flow to complete...
⠦ Waiting for token flow to complete...
⠧ Waiting for token flow to complete...
⠇ Waiting for token flow to complete...
⠏ Waiting

┌─ Error: Permission denied ──────────────────────────────────────────────────┐
│ Token flow did not complete in time                                         │
└─────────────────────────────────────────────────────────────────────────────┘


In [13]:
app = modal.App("ace-rag")

In [14]:
embedding_service = EmbeddingService()

In [15]:
import modal

EmbeddingServiceRemote = modal.Cls.from_name(
    "ace-rag",
    "EmbeddingService",
)

embedding_service = EmbeddingServiceRemote()

In [16]:
test_texts = [
    "What is the purpose of the ACE File Output node?",
    "What properties can be configured on the File Input node?",
]

test_vectors = embedding_service.embed_texts.remote(
    test_texts,
    input_type="query",
)

print("Number of vectors:", len(test_vectors))
print("Embedding dimensions:", len(test_vectors[0]))
print("Vector sample:", test_vectors[0][:5])

Number of vectors: 2
Embedding dimensions: 4096
Vector sample: [0.00656890869140625, 0.0081939697265625, 0.0018053054809570312, 0.00705718994140625, -0.003437042236328125]


In [20]:
for i, vector in enumerate(test_vectors):
    norm = np.linalg.norm(vector)
    assert np.isclose(norm, 1.0, atol=5e-4)

print("All embedding checks passed.")

All embedding checks passed.


In [18]:
import numpy as np

for i, vector in enumerate(test_vectors):
    vector = np.asarray(vector, dtype=np.float64)

    print(f"\nVector {i + 1}")
    print("Dimensions:", vector.shape)
    print("Norm:", np.linalg.norm(vector))
    print("Minimum:", vector.min())
    print("Maximum:", vector.max())
    print("Contains NaN:", np.isnan(vector).any())
    print("Contains Inf:", np.isinf(vector).any())


Vector 1
Dimensions: (4096,)
Norm: 1.0001513478391586
Minimum: -0.09912109375
Maximum: 0.0836181640625
Contains NaN: False
Contains Inf: False

Vector 2
Dimensions: (4096,)
Norm: 1.000014127334102
Minimum: -0.078369140625
Maximum: 0.07904052734375
Contains NaN: False
Contains Inf: False


In [19]:
for i, vector in enumerate(test_vectors):
    norm = np.linalg.norm(vector)
    print(f"Vector {i + 1}: {norm:.10f}")

Vector 1: 1.0001513478
Vector 2: 1.0000141273
